In [1]:
import xarray as xr
import glob
import pickle

%load_ext autoreload
%autoreload 2

# Process zonal data from MESM
Process data by taking the ensemble mean from the 30 member initial condition ensemble. Output this in a numpy format rather than netcdf

In [2]:
experiment_labels = ['CS3','optimized','Tier 1','Tier 2','DECK']
scenarios = {'CS3':['AA','CT'],
             'optimized':['opt_all','opt_all_sine','opt_CS3','opt_DECK','opt_tier1','opt_tier2'],
             'Tier 1':['H-ext','historical','L','M','ML','VLHO','VLLO-ext'],
             'Tier 2':['H-ext-OS','M-ext','ML-ext','L-ext','VLHO-ext'],
             'DECK':['1%-CO2']}

In [4]:
for exp_label in experiment_labels:
  for scen in scenarios[exp_label]:
    # Modify this depending on the data storage location
    path = f'data/MESM/emis_driven/zonal_data/{exp_label}/ZONALANN.{scen}*.nc'
    files = sorted(glob.glob(path))
    ds = xr.open_mfdataset(path, combine='nested', concat_dim='member', parallel=True, coords='minimal')
    ensemble_mean = ds['DT2M'].mean(dim='member')
    ensemble_mean_np = ensemble_mean.compute().values

    # Modify this depending on the output location
    path_np = f'data/MESM/emis_driven/zonal_data_mean/{exp_label}/{scen}_mean.pkl'
    with open(path_np, "wb") as f:
            pickle.dump(ensemble_mean_np, f)